# Solutions · Chapter 04-03 · Splitting I

Worked answers for `notebooks/04_workflow/04-03_splitting_basics.ipynb`.

E8, E9 and E12 all produce results that argue against the tidy version of the lesson. Those are the ones
to read.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")


# SYNTHETIC. The gym panel from 04-01, framed as in 04-03.
def load_members():
    rng = np.random.default_rng(41)
    n = 600
    join_month = rng.integers(1, 13, n)
    commitment = rng.beta(2.0, 2.0, n)
    rows = []
    for member in range(n):
        drifting = 0.0
        base_visits = 2 + 10 * commitment[member]
        for month in range(join_month[member], 25):
            drifting += rng.normal(0.0, 0.35)
            visits = max(0, int(round(rng.poisson(max(0.2, base_visits - drifting)))))
            tickets = int(rng.random() < 0.05 + 0.10 * (visits == 0))
            hazard = 1 / (1 + np.exp(3.0 + 2.5 * commitment[member] - 0.55 * max(0, 4 - visits)))
            cancelled = int(rng.random() < hazard)
            rows.append((member + 1, month, visits, tickets, cancelled))
            if cancelled:
                break
    return pd.DataFrame(rows, columns=["member_id", "month", "visits", "tickets", "cancelled"])


panel = load_members().sort_values(["member_id", "month"])

CUT, HORIZON = 12, 6
active = panel[(panel.month == CUT) & (panel.cancelled == 0)].member_id.unique()
history = panel[panel.member_id.isin(active) & (panel.month <= CUT)]
future = panel[panel.member_id.isin(active) & (panel.month > CUT)]
left = future[(future.month <= CUT + HORIZON) & (future.cancelled == 1)].member_id.unique()

churn = pd.Series(np.isin(active, left).astype(int), index=active)
features = pd.DataFrame({
    "visits_at_cut": history[history.month == CUT].set_index("member_id").visits.reindex(active),
    "mean_visits_last_3": history[history.month > CUT - 3].groupby("member_id").visits.mean().reindex(active),
    "tenure_months": history.groupby("member_id").size().reindex(active),
    "tickets_so_far": history.groupby("member_id").tickets.sum().reindex(active),
})

panel["next_visits"] = panel.groupby("member_id").visits.shift(-1)
visits_table = panel.dropna(subset=["next_visits"]).copy()
visits_table["mean_so_far"] = (visits_table.groupby("member_id").visits
                               .expanding().mean().reset_index(level=0, drop=True))
regression_columns = ["visits", "mean_so_far", "tickets", "month"]

print("%d members, %d cancel; %d member-months" % (len(churn), churn.sum(), len(visits_table)))

## Quick understanding

### E1

| Set | For | Looks allowed |
|---|---|---|
| **Training** | fitting the model's parameters | unlimited - the model is supposed to see it |
| **Validation** | choosing between models, features and hyperparameters | unlimited, but every look inflates its score, so it is not a performance estimate |
| **Test** | estimating what the chosen model will do on new data | **once**, at the very end, after every choice is final |

### E2

Because it measures a different quantity. A model's error on the rows it was fitted to is a measure of
**how well it can reproduce data it has already seen**, and that can be driven to zero by anyone willing
to add capacity - the unlimited-depth tree reached 0.3453 while scoring 3.4772 on unseen rows.

An estimate can be biased and still be an estimate; you can correct a known bias. This is not biased, it
is **a measurement of something else**, and there is no correction because the quantity it measures does
not depend on generalisation at all.

### E3

**Fixes:** the class balance of each split. Stratified, every held-out set here is **13.38%** cancellers,
with a standard deviation of exactly **0.0000**.

**Leaves untouched:** which particular members land on each side. The 200-split AUC spread in Part 2 was
already stratified and still ran **0.6080 to 0.8185**, standard deviation 0.0425.

The two numbers are 0.0000 and 0.0425, and the second is the one that dominates a reported score.

## Hand calculation

### E4

The test set is 100 rows. The positive rate is `24/400` = 6%, so the expected number of positives is
`100 x 0.06` = **6**.

With a standard deviation of about 2.1, a two-standard-deviation range is roughly `6 +/- 4.2`, so
**anywhere from about 2 to about 10** would be unsurprising.

**What that implies:** every metric on that test set is computed from a handful of positives, and the
handful is itself random. Recall moves in steps of 1/6 to 1/10 - a single positive classified differently
swings it by 10 to 17 percentage points. **A test set of 100 rows with 6 positives is a test set of size
6 for every purpose that matters**, and no amount of careful modelling makes a metric computed from it
precise. This problem needs cross-validation, or more data, or both.

### E5

Accuracy = `120/150` = **0.800**.

- `n = 150`: standard error `sqrt(0.8 x 0.2 / 150)` = **0.0327**. Two standard errors gives roughly
  **0.735 to 0.865** - a range 13 percentage points wide.
- `n = 1500`: standard error **0.0103**. Range roughly **0.779 to 0.821**, 4 points wide.

**Ten times the data shrank the interval by a factor of about 3.2, not 10** - the square root of 10, as
03-02 requires. To *halve* the interval you need **four times** the data, so 600 rows would have done it.

The practical reading: at 150 test rows you cannot tell 80% from 85%. Reporting "80.0%" from that test
set states three digits of which one is real.

### E6

The winner's reported score is the true score plus the maximum of 20 noise draws. The expected maximum of
20 standard normals is about 1.87, and each draw has standard deviation 0.04, so the expected inflation is
`1.87 x 0.04` = **about 0.075 AUC points**.

That is enormous - roughly the entire real difference between a useful model and a mediocre one in this
chapter. And it requires no incompetence whatsoever: twenty honest experiments, each scored on held-out
data, reported truthfully.

**The number of things you tried is part of the result**, in exactly the way the number of hypotheses
tested is part of a p-value. Chapter 02-07 made this point about screening columns; it is the same
arithmetic.

### E7

The 30% row has standard deviation 0.0425 on 157 test rows. Under the square-root law the standard
deviation scales as `1/sqrt(n)`, so to reach 0.01 you need

`157 x (0.0425 / 0.01)^2` = **about 2,800 test rows**.

**There are 521 members in total.** So it is not achievable at any split - not by holding out more, not by
holding out everything. The precision this dataset can support is about 0.04 of AUC, and any comparison
finer than that requires either more members or a method that reuses rows, which is cross-validation.

Worth noticing that this calculation is available **before** any modelling, from the sample size alone.
It tells you what questions the data can answer, and that is a framing question (04-01) as much as a
splitting one.

In [ ]:
print("E4  expected positives in a 100-row test set: %.1f" % (400 * 0.06 * 0.25))
print("    roughly %.1f to %.1f at two standard deviations" % (6 - 2 * 2.1, 6 + 2 * 2.1))
print()
for rows in [150, 600, 1500]:
    error = np.sqrt(0.8 * 0.2 / rows)
    print("E5  n=%4d  accuracy 0.800  se %.4f  ->  %.3f to %.3f  (width %.3f)"
          % (rows, error, 0.8 - 2 * error, 0.8 + 2 * error, 4 * error))
print()
print("E6  expected inflation of the best of 20: %.4f AUC" % (1.87 * 0.04))
print("E7  test rows needed for sd 0.01: %.0f   (the dataset has %d members)"
      % (157 * (0.0425 / 0.01) ** 2, len(churn)))

## Coding

### E8 - is the model distinguishable from the one rule?

In [ ]:
def split_spread(scorer, n_splits=200):
    values = np.array([scorer(seed) for seed in range(n_splits)])
    return {"mean": round(values.mean(), 4), "sd": round(values.std(), 4),
            "5th": round(np.percentile(values, 5), 4), "95th": round(np.percentile(values, 95), 4)}, values


def logistic_on_split(seed, stratify=True):
    train_X, test_X, train_y, test_y = train_test_split(
        features, churn, test_size=0.3, random_state=seed,
        stratify=churn if stratify else None)
    centre, spread = train_X.mean(), train_X.std()
    model = LogisticRegression(max_iter=2000).fit((train_X - centre) / spread, train_y)
    return roc_auc_score(test_y, model.predict_proba((test_X - centre) / spread)[:, 1])


def one_rule_on_split(seed):
    train_X, test_X, train_y, test_y = train_test_split(
        features, churn, test_size=0.3, random_state=seed, stratify=churn)
    _, threshold = max((roc_auc_score(train_y, (train_X.mean_visits_last_3 <= t).astype(int)), t)
                       for t in np.unique(train_X.mean_visits_last_3))
    return roc_auc_score(test_y, (test_X.mean_visits_last_3 <= threshold).astype(int))


logistic_summary, logistic_values = split_spread(logistic_on_split)
rule_summary, rule_values = split_spread(one_rule_on_split)
print(pd.DataFrame([dict(model="logistic regression", **logistic_summary),
                    dict(model="best one-rule threshold", **rule_summary)]).to_string(index=False))

difference = logistic_values - rule_values
print()
print("paired on the same splits: mean difference %+.4f, sd %.4f"
      % (difference.mean(), difference.std())) 
print("the logistic wins on %.1f%% of the 200 splits" % (100 * np.mean(difference > 0)))

**No, they are not distinguishable.** The logistic averages 0.7220 and the rule 0.7146 - a gap of 0.0074
against a spread of 0.0425 each.

The right way to compare is **paired**, on the same splits, which removes the split-to-split variation
common to both. Even then the mean difference is **+0.0074 with a standard deviation of 0.0339**: the
difference is a fifth of its own noise, and the logistic wins on **58% of splits**, barely better than a
coin.

Note how much the pairing helped, though: the unpaired standard deviations are 0.0425 and 0.0414, while
the paired difference has 0.0339. Comparing two models on the *same* splits is strictly better than
comparing two independently-obtained numbers, and it costs nothing. **When you compare models, hold the
splits fixed.**

The conclusion stands from 04-02: the model is worth about one threshold on one column, and this is the
measurement that entitles you to say so.

### E9 - does stratification narrow the spread?

In [ ]:
stratified_summary, _ = split_spread(logistic_on_split)
plain_summary, _ = split_spread(lambda seed: logistic_on_split(seed, stratify=False))
print(pd.DataFrame([dict(split="stratified", **stratified_summary),
                    dict(split="plain random", **plain_summary)]).to_string(index=False))

**No - the standard deviations are 0.0425 either way, identical to four decimals.**

That is not what most people predict, and it does not contradict the chapter. Stratification eliminates
one source of variation completely (the test set's base rate, sd 0.0226 to 0.0000) and that source turns
out to be **a small contributor to the variation in AUC**. The dominant term is which particular members
land in the test set - whether the twenty-one cancellers you happen to hold out are easy or hard ones -
and stratification does nothing about that because it only balances counts, not difficulty.

The 5th-95th range does widen slightly without stratification (0.6482-0.7936 against 0.6430-0.7816), so
the tails are a little worse behaved, which is consistent with an extra small variance component.

**Use stratification anyway.** The argument for it is not variance reduction; it is that it makes the
denominator of every metric identical across experiments, so two numbers are comparable and a metric like
accuracy is measured against a fixed base rate rather than a moving one. And on a dataset with, say, 15
positives rather than 70, a plain split can hand you a test set with two - which stratification prevents
and which no amount of averaging repairs.

### E10 - does the best depth survive a different split?

In [ ]:
depths = [1, 2, 3, 5, 8, 12, 16, 20]
fig, ax = plt.subplots(figsize=(8, 4.6))
rows = []
for seed in range(5):
    fit_rows, holdout_rows = train_test_split(visits_table, test_size=0.3, random_state=seed)
    curve = []
    for depth in depths:
        tree = DecisionTreeRegressor(max_depth=depth, random_state=0)
        tree.fit(fit_rows[regression_columns], fit_rows.next_visits)
        curve.append(mean_absolute_error(holdout_rows.next_visits,
                                         tree.predict(holdout_rows[regression_columns])))
    ax.plot(depths, curve, "o-", alpha=0.8, label="split %d" % seed)
    rows.append({"split": seed, "best depth": depths[int(np.argmin(curve))],
                 "its MAE": round(min(curve), 4),
                 "MAE at depth 3": round(curve[2], 4), "MAE at depth 5": round(curve[3], 4)})
ax.set_xlabel("maximum tree depth")
ax.set_ylabel("MAE on unseen rows")
ax.set_title("Five splits, five curves, mostly the same answer")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(pd.DataFrame(rows).to_string(index=False))

**Depth 3 wins on four splits out of five; split 3 prefers depth 5.**

So the answer is a qualified yes, and the qualification is the lesson. The *level* of the curve moves a
lot between splits - the best MAE ranges from 2.3669 to 2.4535, which is larger than the entire gap
between depth 3 and depth 5 on any single curve. The *shape* is stable: every split agrees that the
minimum is somewhere around 3 to 5 and that depth 12 and beyond is bad.

**A hyperparameter chosen from a single split is reliable only to the resolution the split can support**,
and here that resolution is "somewhere between 3 and 5", not "3". Choosing 3 because one split said so is
over-reading; choosing something in the 3-5 region is well supported.

That is exactly the situation cross-validation is for - averaging the five curves gives a single curve
whose minimum is meaningful, at the cost of five fits instead of one (04-07).

### E11 - a search that reports honestly

In [ ]:
def honest_search(candidate_columns, warn_above=0.03):
    results = []
    for chosen in candidate_columns:
        centre, spread = train_X[chosen].mean(), train_X[chosen].std()
        model = LogisticRegression(max_iter=2000).fit((train_X[chosen] - centre) / spread, train_y)
        results.append((roc_auc_score(val_y, model.predict_proba((val_X[chosen] - centre) / spread)[:, 1]),
                        roc_auc_score(test_y, model.predict_proba((test_X[chosen] - centre) / spread)[:, 1]),
                        chosen))
    on_validation, on_test, winner = max(results, key=lambda row: row[0])
    print("tried %d candidates" % len(results))
    print("  winner's validation AUC : %.4f   <- do not report this" % on_validation)
    print("  winner's test AUC       : %.4f   <- report this" % on_test)
    if on_validation - on_test > warn_above:
        print("  WARNING: selection premium %.4f exceeds %.2f - the search is fitting the "
              "validation set" % (on_validation - on_test, warn_above))
    return winner, on_test


noise_rng = np.random.default_rng(7)
wide = features.copy()
for j in range(20):
    wide["noise_%d" % j] = noise_rng.normal(size=len(features))

train_X, rest_X, train_y, rest_y = train_test_split(wide, churn, test_size=0.4,
                                                    random_state=0, stratify=churn)
val_X, test_X, val_y, test_y = train_test_split(rest_X, rest_y, test_size=0.5,
                                                random_state=0, stratify=rest_y)

candidate_columns = [list(noise_rng.choice(wide.columns, size=4, replace=False)) for _ in range(40)]
honest_search(candidate_columns)

**Justifying the threshold.** 0.03 is roughly the standard error of an AUC on a validation set this size
- Part 2 measured 0.0425 on 157 rows, and the validation set here is 104. A premium *smaller* than that is
indistinguishable from ordinary split noise and warns about nothing. A premium *larger* than it means the
selection has moved the score by more than chance would, which is the definition of fitting the validation
set.

So the threshold is not a convention, it is **the noise floor of the set you are selecting on**, and it
should be computed rather than assumed - which `split_spread` from E8 already does.

### E12 - more noise columns, more candidates

In [ ]:
wider_rng = np.random.default_rng(7)
wider = features.copy()
for j in range(100):
    wider["noise_%d" % j] = wider_rng.normal(size=len(features))

train_X, rest_X, train_y, rest_y = train_test_split(wider, churn, test_size=0.4,
                                                    random_state=0, stratify=churn)
val_X, test_X, val_y, test_y = train_test_split(rest_X, rest_y, test_size=0.5,
                                                random_state=0, stratify=rest_y)

many = [list(wider_rng.choice(wider.columns, size=4, replace=False)) for _ in range(200)]
winner, _ = honest_search(many)
print()
print("  the winning columns:", ", ".join(winner))
print("  of which genuinely informative: %d of 4"
      % sum(1 for column in winner if not column.startswith("noise")))

**The premium grows to 0.1315 and the test score does not improve - it falls slightly**, from 0.6366 with
20 noise columns and 40 candidates to 0.6224 with 100 and 200.

And look at what won: **three noise columns and one real one.** Two hundred draws found a combination of
random numbers that happens to separate 104 validation members well, and that is all it found.

This is the whole hazard of a large search, in a form small enough to see. More candidates does not mean
a better model; it means a better *search over the validation set*, and past a point those are opposite
things. The reported number goes up, the delivered number goes down, and nothing in the process signals
which is happening - except a held-out test set that was never part of the search.

## Interpretation

### E13

At 90% accuracy on 2,000 rows the standard error is `sqrt(0.9 x 0.1 / 2000)` = **0.0067**, or **0.67
percentage points**.

A claimed improvement of **0.4 points is less than one standard error** of either number, so on the face
of it the result is inside the noise.

What you would want to know, in order:

1. **Is the comparison paired?** If both methods were evaluated on the *same* test set, the relevant
   quantity is the standard error of the *difference*, which is smaller than 0.67 points because the two
   share the test set's difficulty - exactly the effect E8 measured. A paired 0.4-point difference can be
   real. An unpaired one almost certainly is not.
2. **How many variants were tried before this one?** E6's arithmetic applies: the best of twenty honest
   attempts is inflated by about 1.9 standard errors, which is 1.3 points here - three times the claimed
   improvement.
3. **How many times has this benchmark been used, by everybody?** A public test set that a field has
   optimised against for years is a validation set wearing a test set's name, and the community-wide
   selection premium is unmeasurable and large.
4. **Does it replicate on a different test set?** The only answer that settles it.

### E14

> "The test score is the only one of the two that estimates what the model will do on new data - the
> validation score is high partly *because* we chose this model for scoring high on it, so it is not an
> estimate of anything. Being smaller and noisier makes the test number less precise, which is an argument
> for a bigger test set or for cross-validation, not for reporting the other number instead. If we want a
> tighter estimate I would rather re-split several times and report the range than report 0.91."

The colleague's reasoning inverts the relationship: **noisier is not the same as biased.** The 0.91 is
precise and systematically too high; the 0.88 is noisy and unbiased. You can fix noise with more data;
you cannot fix selection bias with anything except a set that had no part in the selection.

## Debugging

### E15

**Explanation 1 - it is chance.** With a validation set of 104 and a test set of 105, both scores have a
standard error around 0.04, so a gap of a few points either way is unremarkable. "Consistently, across
several models" is weaker evidence than it sounds, because those models share the same two sets - if the
test set happens to be the easier of the two, *every* model will score higher on it.

**Explanation 2 - the two sets are not exchangeable.** The test set is genuinely easier: different time
period, different mix of members, a different base rate if the split was not stratified.

**The check that distinguishes them: re-split many times and see whether the sign persists.** If test
beats validation on roughly half of fresh random splits, it was chance and the original assignment was
lucky. If test beats validation on almost every fresh split of the same underlying data, then the sets
differ systematically and you should find out how - starting with the base rate and the date range of
each.

### E16

**The most likely cause is that the rows are not independent** - the same member, or the same time
period, appears on both sides of the split, so the "held-out" set was not held out in any meaningful
sense.

This chapter's discipline did not prevent it because **every rule in it was followed.** The split was
random, stratified, and never inspected. All three are correct, and all three assume something this
chapter never checked: that one row is one independent observation. On a member-month table it is not -
04-01's E18 found the average member appearing about eleven times - and a random split of such rows tests
whether the model recognises members it has already met.

The second candidate, if the data is temporal: the test set contains rows from *before* some training
rows, so the model saw the future during training and cannot in production.

**Both are 04-04.** The point of ending on this exercise is that a correct random split is not a safe
random split, and nothing in the score warns you.

## Exam and interview reasoning

### E17

> "The test set answers 'how good is the model I picked'. The moment I use it to *do* the picking, it
> stops answering that, because the winner's score includes whatever luck it had on those particular
> rows. In this chapter, choosing the best of forty candidates gave a validation score of 0.75 and a test
> score of 0.64 - a tenth of an AUC that was pure selection. So I need one set to choose on and a separate
> one, untouched, to report from."

**"We only have 400 rows - what do you do instead?"**

> "Cross-validation for the choosing, and one held-out test set kept for the end. Carving out a fixed
> validation set from 400 rows leaves it too small to choose reliably - here a 104-row validation set
> produced a 0.12 selection premium - whereas cross-validation uses every row for both training and
> validation across the folds, so the estimate is far more stable at the cost of fitting k times. I would
> also report the fold-to-fold spread rather than a single number, because at this sample size the spread
> is the honest headline."

## Transfer to a different situation

### E18

**Split by patient, not by image.** All 50,000 images from one patient go entirely to one side.

**Stratify on the diagnosis** - and, if the classes are unevenly distributed across sites, scanners or
time periods, on those too, since a model can learn the scanner instead of the disease.

**The mistake a plain random split of the 50,000 images causes:** images from the same patient land on
both sides. The model learns to recognise *that patient* - their anatomy, their positioning, their
scanner's artefacts - and is then rewarded at test time for recognising them again. The reported accuracy
measures patient re-identification, and the model fails on the first genuinely new patient. With about 167
images per patient, essentially every test patient would also be a training patient.

**How to detect it if somebody has already done it:** re-split by patient and re-score, changing nothing
else. A large drop is the diagnosis. Cheaper still, check directly whether any patient identifier appears
in both sets - one line, and it should be an assertion in the pipeline rather than a thing you remember to
check.

## Explain it to someone non-technical

### E19

> "Suppose I let you take a driving test forty times and then reported your best attempt. That number
> tells you how you drive on a lucky day, not how you drive. To know how you actually drive, I have to
> watch one attempt I did not get to choose. It is the same with the model: while I am developing it I try
> many versions and keep the best, so that best score is partly luck. I keep a set of data locked away and
> only look at it once, at the end, so I have one number that luck could not have chosen."

(93 words.)

## Optional challenge

### E20 - the selection-premium curve

In [ ]:
premium_rng = np.random.default_rng(11)
pool = []
for _ in range(400):
    chosen = list(premium_rng.choice(wider.columns, size=4, replace=False))
    centre, spread = train_X[chosen].mean(), train_X[chosen].std()
    model = LogisticRegression(max_iter=2000).fit((train_X[chosen] - centre) / spread, train_y)
    pool.append((roc_auc_score(val_y, model.predict_proba((val_X[chosen] - centre) / spread)[:, 1]),
                 roc_auc_score(test_y, model.predict_proba((test_X[chosen] - centre) / spread)[:, 1])))
pool = np.array(pool)

sizes = [1, 2, 5, 10, 20, 50, 100, 200]
draw_rng = np.random.default_rng(5)
curve = []
for k in sizes:
    premiums, validations, tests = [], [], []
    for _ in range(200):
        drawn = pool[draw_rng.choice(len(pool), size=k, replace=False)]
        picked = drawn[np.argmax(drawn[:, 0])]
        validations.append(picked[0])
        tests.append(picked[1])
        premiums.append(picked[0] - picked[1])
    curve.append({"candidates": k, "validation": round(float(np.mean(validations)), 4),
                  "test": round(float(np.mean(tests)), 4),
                  "premium": round(float(np.mean(premiums)), 4)})
curve = pd.DataFrame(curve)
print(curve.to_string(index=False))

fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.6))

left.plot(curve.candidates, curve.validation, "o-", color="#D55E00", label="best validation score")
left.plot(curve.candidates, curve.test, "s-", color="#0072B2", label="what it really scores")
left.set_xscale("log")
left.set_xlabel("candidates tried (log scale)")
left.set_ylabel("AUC")
left.set_title("Both rise - but not together")
left.legend(fontsize=8)

right.plot(curve.candidates, curve.premium, "o-", color="#000000")
right.axhline(curve.premium.iloc[0], color="#999999", linestyle=":",
              label="the k=1 offset: not selection at all")
right.set_xscale("log")
right.set_ylim(0, 0.15)
right.set_xlabel("candidates tried (log scale)")
right.set_ylabel("validation AUC minus test AUC")
right.set_title("The premium saturates early")
right.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Not a logarithm. It rises steeply to about five candidates and then nearly flattens** - 0.1127 at
five, 0.1301 at two hundred, a rise of less than two points across a fortyfold increase in searching.

That is not the answer the question expected, and disentangling it is the useful part. **Two different
things are inside that gap, and only one of them is selection.**

**The constant offset.** At `k = 1` the premium is already **0.0281**, and with one candidate there is
nothing to select. That 0.028 is simply the fact that this particular test set is harder than this
particular validation set - 105 rows against 104, split by chance. It is a fixed handicap sitting under
the whole curve, and it is the same split lottery as Part 2 in a different costume.

**The selection term** is what remains, and it does grow with `k`, but it approaches a ceiling. It cannot
exceed the amount of noise available to exploit, and with 104 validation rows there is only so much. The
theoretical `sqrt(2 log k)` growth of an extreme value applies to unbounded noise; a bounded metric on a
small set saturates instead.

**Now read the left panel, which is the finding.** Both curves rise: validation from 0.5408 to 0.7609,
and test from **0.5127 to 0.6308**. Unlike the chapter's version, the test score here genuinely improves
with more searching - because with only four real columns among 104, a draw of five candidates usually
contains no informative column at all. **Early candidates buy real discovery; later ones buy mostly
premium.** The crossover is around ten to twenty, after which test gains 0.06 while validation gains 0.09.

So the answer for a 1,000-configuration search is more nuanced than "it is all inflation":

- **The first few dozen configurations do real work** if your search space contains genuinely different
  ideas. Searching more than that mostly refines your description of the validation set.
- **The premium does not grow without limit, but it does not shrink either** - it arrives early and stays,
  so a large search is not self-correcting.
- **Report the test score and the number of configurations tried.** The second is part of the first.
- **The validation set has a capacity.** 104 rows can separate a handful of models, not a thousand. If you
  need to search widely, you need cross-validation for the choosing - 04-07.

And the general habit this exercise is really teaching: when a curve does not look like the shape you
predicted, **decompose it before explaining it.** The k=1 point was the whole key here, and it is the
point most people would not have plotted.